### Import Libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

### Load Dataset

In [ ]:
df = pd.read_csv("../data/raw/student_depression.csv")
df.head()

### Dataset Overview

In [ ]:
df.info()
df.describe(include="all")

# 1) EDA

### Check for Duplicates

In [ ]:
# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

### Check for Missing Values

In [ ]:
# Kiểm tra các cột có giá trị null
print(df.isnull().sum())

### Handle Missing Values

In [ ]:
# Impute the three missing Financial Stress values with the median
df['Financial Stress'] = df['Financial Stress'].fillna(df['Financial Stress'].median())

In [ ]:
print(df.isnull().sum())

# 2) Reprocessing data

### Map Sleep Duration to Numeric Values

In [ ]:
# Mapping thời lượng ngủ thành số
sleep_map = {"Less than 5 hours": 4, "5-6 hours": 5.5, "7-8 hours": 7.5, "More than 8 hours": 9}
df["Sleep Duration"] = df["Sleep Duration"].map(sleep_map)

# Kiểm tra và xử lý NaN
if df["Sleep Duration"].isnull().sum() > 0:
    print("Found NaN in 'Sleep Duration' after mapping.")
    df["Sleep Duration"] = df["Sleep Duration"].fillna(df["Sleep Duration"].median())

### Drop Unnecessary Columns

In [ ]:
df = df.drop(columns=["id", "City", "Profession"])

### Identify Categorical Columns

In [ ]:
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical Columns:", categorical_columns)

### One-Hot Encode Categorical Columns

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
# 1. Đầu tiên chia dữ liệu
X = df.drop("Depression", axis=1)
y = df["Depression"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Xác định các cột cần scale
columns_to_scale = [
    "Age",
    "Academic Pressure",
    "Work Pressure",
    "CGPA",
    "Study Satisfaction",
    "Job Satisfaction",
    "Work/Study Hours",
    "Financial Stress",
    "Sleep Duration",
]

# 3. Khởi tạo scaler và fit_transform chỉ trên tập train
scaler = MinMaxScaler()
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])

# 4. Transform tập test bằng scaler đã fit trên tập train
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

In [ ]:
train_data = X_train.copy()
test_data = X_test.copy()

# Thêm cột Depression (target feature)
train_data["Depression"] = y_train
test_data["Depression"] = y_test

# Lưu ra file CSV
train_data.to_csv("../data/processed/train.csv", index=False)
test_data.to_csv("../data/processed/test.csv", index=False)

print("Saved data/processed/train.csv and data/processed/test.csv")
print(f"Kích thước tập train: {train_data.shape}")
print(f"Kích thước tập test: {test_data.shape}")

# 3) Feature engineering

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Lấy tầm quan trọng của các đặc trưng
importances = rf.feature_importances_
feature_names = X_train.columns

# Sắp xếp và hiển thị các đặc trưng quan trọng nhất
feature_importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
feature_importance_df = feature_importance_df.sort_values(by="Importance", ascending=False)
print(feature_importance_df)

# Vẽ biểu đồ trực quan hóa tầm quan trọng
plt.figure(figsize=(15, 10))
sns.barplot(x="Importance", y="Feature", data=feature_importance_df)
plt.title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
coefs = log_reg.coef_[0]
feature_names = X_train.columns

# Tạo DataFrame để xem độ quan trọng
coef_df = pd.DataFrame({"Feature": feature_names, "Coef": coefs})
coef_df["AbsCoef"] = coef_df["Coef"].abs()
coef_df = coef_df.sort_values(by="AbsCoef", ascending=False)

print("Top 10 đặc trưng quan trọng nhất theo Logistic Regression:")
print(coef_df)

# Vẽ biểu đồ trực quan hóa hệ số
plt.figure(figsize=(15, 10))
sns.barplot(x="AbsCoef", y="Feature", data=coef_df)
plt.title("Absolute Feature Coefficients (Logistic Regression)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# Đã huấn luyện log_reg hoặc rf rồi
result = permutation_importance(log_reg, X_train, y_train, n_repeats=10, random_state=42)

perm_importance_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": result.importances_mean,
        "Std": result.importances_std,
    }
).sort_values(by="Importance", ascending=False)

print("Top 10 đặc trưng quan trọng nhất theo Permutation Importance:")
print(perm_importance_df)

plt.figure(figsize=(15, 10))
sns.barplot(x="Importance", y="Feature", data=perm_importance_df)
plt.title("Top 20 Permutation Importances (Logistic Regression)")
plt.tight_layout()
plt.show()